In [0]:
from pyspark.sql import functions as F
start_date = "2020-01-01"
end_date = "2025-12-31"

date_gold = spark\
.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as date_key")

dim_date = date_gold.select(
    F.col("date_key"), 
    F.year("date_key").alias("year"),
    F.month("date_key").alias("month"),
    F.date_format("date_key", "MMMM").alias("month_name"), 
    F.dayofmonth("date_key").alias("day"),
    # transforming the date to the first day of the month(15/05/2023 -> 01/05/2023)so we can join with fact_sales
    F.trunc("date_key", "MM").alias("month_start_date"),
)
(dim_date.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true") # if we want to change the columns later
  .saveAsTable("sales_lakehouse.gold.dim_date"))


In [0]:
dim_date.printSchema()